# AE Pretraining Ablation Runner

Runs Stage 1 and optional Stage 1B experiments only. This notebook compares AE reconstruction quality before spending time on Stage 3/4/5 downstream runs.

In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys

# Colab/bootstrap settings. Set GITHUB_REPO_URL to your fork if needed.
GITHUB_REPO_URL = "https://github.com/neurovlm/neurovlm.git"
GITHUB_BRANCH = "neurovlm_gnn"
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/neurovlm")
LOCAL_PROJECT_DIR = Path("/content/neurovlm")
PULL_LATEST = False

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "experiments" / "3dcnn" / "atlas_free_cnn").exists():
            return candidate
    return None

REPO_ROOT = find_repo_root(Path.cwd())

if REPO_ROOT is None and IN_COLAB:
    target = DRIVE_PROJECT_DIR if USE_GOOGLE_DRIVE else LOCAL_PROJECT_DIR
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, "--single-branch", GITHUB_REPO_URL, str(target)], check=True)
    REPO_ROOT = target

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not find the neurovlm repo root. In Colab, set GITHUB_REPO_URL and run this cell to clone it; "
        "locally, start Jupyter from the repository root."
    )

subprocess.run(["git", "checkout", GITHUB_BRANCH], cwd=REPO_ROOT, check=True)

if PULL_LATEST:
    subprocess.run(["git", "pull", "--ff-only", "origin", GITHUB_BRANCH], cwd=REPO_ROOT, check=True)

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "experiments" / "3dcnn"))
sys.path.insert(0, str(REPO_ROOT / "src"))
print(f"Using repo root: {REPO_ROOT}")

from atlas_free_cnn.training.train_autoencoder import train_from_config, train_stage1b_from_config
from atlas_free_cnn.pipeline_outputs import create_full_pipeline_run_dir, write_table


In [ ]:
TRAIN_JSONL = "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild/splits/train.jsonl"
VAL_JSONL = "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild/splits/val.jsonl"
TEST_JSONL = "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild/splits/test.jsonl"
AE_CHECKPOINT_SELECTION = "best_top5_dice"
RUN_STAGE1B = True
paths = create_full_pipeline_run_dir("runs_atlas_free_cnn_ae_ablation", prefix="ae_ablation")
RUN_DIR = Path(paths["run_dir"])
print(RUN_DIR)

In [ ]:
recipes = [
    {
        "ae_variant": "mixed_baseline_raw_mse",
        "ae_training_recipe": "baseline_raw_mse",
        "source_sampling": "natural",
        "loss": {"type": "raw_mse", "lambda_foreground": 0.0, "lambda_topk": 0.0, "prediction_activation": "none"},
        "checkpoint_selection_metric": "best_val_loss",
    },
    {
        "ae_variant": "mixed_balanced_raw_mse",
        "ae_training_recipe": "mixed_balanced_raw_mse",
        "source_sampling": "balanced",
        "loss": {"type": "raw_mse", "prediction_activation": "none"},
        "checkpoint_selection_metric": "best_top5_dice",
    },
    {
        "ae_variant": "mixed_balanced_hybrid_loss",
        "ae_training_recipe": "mixed_balanced_hybrid_loss",
        "source_sampling": "balanced",
        "loss": {"type": "hybrid_recon", "lambda_foreground": 0.10, "lambda_topk": 0.05, "topk_percent": 5, "prediction_activation": "none"},
        "checkpoint_selection_metric": "best_top5_dice",
    },
]

In [ ]:
results = []
for recipe in recipes:
    out_dir = RUN_DIR / "01_stage1_ae_pretraining" / recipe["ae_variant"]
    cfg = {
        "train_jsonl": TRAIN_JSONL,
        "val_jsonl": VAL_JSONL,
        "test_jsonl": TEST_JSONL,
        "output_dir": str(out_dir),
        "checkpoint_dir": str(out_dir / "checkpoints"),
        "data_mode": "mixed",
        "target_shape": [36, 45, 38],
        "model": {"latent_dim": 384, "base_channels": 64, "num_blocks": 4, "dropout": 0.1, "norm": "group", "pooling": "max"},
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "amp": True,
        "gradient_clipping": 1.0,
        "batch_size": 64,
        "epochs": 300,
        **recipe,
    }
    result = train_from_config(cfg)
    results.append({"ae_variant": recipe["ae_variant"], "checkpoint_dir": result["checkpoint_dir"], "best_checkpoint": result["best_checkpoint"]})
results

In [ ]:
if RUN_STAGE1B and results:
    seed_checkpoint = results[-1]["best_checkpoint"]
    for mode, domain in [("mixed_pretrain_to_pubmed", "pubmed"), ("mixed_pretrain_to_statmaps", "statmaps")]:
        out_dir = RUN_DIR / "02_stage1b_ae_finetuning" / domain
        cfg = {
            "stage1b_mode": mode,
            "mixed_pretrain_checkpoint": seed_checkpoint,
            "train_jsonl": TRAIN_JSONL,
            "val_jsonl": VAL_JSONL,
            "test_jsonl": TEST_JSONL,
            "output_dir": str(out_dir),
            "checkpoint_dir": str(out_dir / "checkpoints"),
            "ae_variant": "mixed_to_pubmed" if domain == "pubmed" else "mixed_to_statmaps",
            "lr": 1e-4,
            "freeze_mode": "none",
            "epochs": 100,
            "loss": {"type": "raw_mse", "prediction_activation": "none"},
        }
        ft_result = train_stage1b_from_config(cfg)
        results.append({"ae_variant": cfg["ae_variant"], "checkpoint_dir": ft_result["checkpoint_dir"], "best_checkpoint": ft_result["best_checkpoint"]})

In [ ]:
summary_rows = []
for row in results:
    metrics_path = Path(row["checkpoint_dir"]).parent / "metrics" / "reconstruction_summary_by_source.csv"
    metrics = {}
    if metrics_path.exists():
        with metrics_path.open(newline="") as f:
            for m in csv.DictReader(f):
                if m.get("split") in {"val", "test"} and m.get("source_detail") == "ALL_DETAILS" and m.get("source") in {"pubmed", "neurovault", "nilearn"}:
                    src = m["source"]
                    metrics[f"{src}_spatial_corr"] = m.get("spatial_corr", "")
                    metrics[f"{src}_top5_dice"] = m.get("top5_dice", "")
    values = []
    for key, value in metrics.items():
        try:
            values.append(float(value))
        except Exception:
            pass
    ranking_score = sum(values) / len(values) if values else ""
    summary_rows.append({**row, **metrics, "ranking_score": ranking_score, "metrics_path": str(metrics_path), "recommendation_note": "Prioritize spatial_corr/top5_dice by source; do not rank by MSE alone."})
summary_rows = sorted(summary_rows, key=lambda r: float(r["ranking_score"]) if r["ranking_score"] != "" else -999, reverse=True)
write_table(RUN_DIR / "07_final_comparison" / "best_checkpoints_to_inspect.csv", summary_rows)
write_table(RUN_DIR / "07_final_comparison" / "final_summary_table.csv", summary_rows)
write_table(RUN_DIR / "07_final_comparison" / "ae_ablation_leaderboard.csv", summary_rows)
print("AE ablation summary written to", RUN_DIR / "07_final_comparison")
summary_rows
